In [21]:
from langchain_core.tools import tool
from dotenv import load_dotenv
load_dotenv()

@tool
def get_weather(city: str) -> str:
    """ Get a weather for a city."""
    return f"The weather in {city} is sunny."

In [22]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

model = ChatGroq(model="openai/gpt-oss-120b")
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant."
)

In [23]:
response = agent.invoke({"messages": [{"role": "user", "content": "What is the weather like in New York?"}]})
response["messages"]

[HumanMessage(content='What is the weather like in New York?', additional_kwargs={}, response_metadata={}, id='93dff206-f136-4f82-9e29-6a23305c49fa'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the weather like in New York?" We need to fetch weather using provided function get_weather. Use city: "New York". Then respond with the weather. We\'ll call function.', 'tool_calls': [{'id': 'fc_0c107ee6-c3fa-48e6-857f-30387098e10b', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 138, 'total_tokens': 209, 'completion_time': 0.150293233, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.025563719, 'prompt_tokens_details': None, 'queue_time': 0.358950568, 'total_time': 0.175856952}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6c36ac20de', 'service_tier': 'on_demand', 'finish_reason': 'too

In [24]:
# In Groq, content is directly a string (unlike Gemini which returned a list of dicts)
content = response["messages"][-1].content
print(content if isinstance(content, str) else content[0]['text'])

The weather in New York is sunny.


In [25]:
for chunk in agent.stream({'messages': [{'role': 'user', 'content': 'What is the weather in Boston'}]}):
    print(chunk)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What is the weather in Boston". We need to call get_weather with city "Boston".', 'tool_calls': [{'id': 'fc_0a383069-6639-4edb-8c6a-293e5343ef76', 'function': {'arguments': '{"city":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 135, 'total_tokens': 185, 'completion_time': 0.109136685, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.005736183, 'prompt_tokens_details': None, 'queue_time': 0.401792894, 'total_time': 0.114872868}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0a2d2-28d7-7822-b1f5-f2d5b8a01a4c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'fc_0a383069-6639-4edb-8c6a-293e5343ef76', 'type': '

In [26]:

from IPython.core import events
async for event in agent.astream_events({"messages": [{"role": "user", "content": "What is the weather in Boston"}]}, version="v2"):
    if event["event"] == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)
    elif event["event"] == "on_tool_start":
        print(f"\n[calling tool: {event['name']}]")
    elif event["event"] == "on_tool_end":
        print(f"[tool result: {event['data']['output']}]")


[calling tool: get_weather]
[tool result: content='The weather in Boston is sunny.' name='get_weather' tool_call_id='fc_5c2d6f3b-caaa-43fb-ba48-9d0331a09e40']
The current weather in Boston is sunny.

In [27]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

model = ChatGroq(model="openai/gpt-oss-120b")
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
    checkpointer=InMemorySaver()
)

In [28]:
config= {"configurable": {"thread_id": "converstion-1"}}
r1 = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Boston?"}]}, config)
print(r1["messages"][-1].content)

The current weather in Boston is sunny.


In [29]:
r2 = agent.invoke({"messages": [{"role": "user", "content": "Is it good for a walk?"}]}, config)
print(r2["messages"][-1].content)

A sunny day is generally ideal for a walk—clear skies, plenty of natural light, and usually pleasant temperatures. If the temperature is moderate (not too hot or cold) and there’s no strong wind or rain forecast, it should be a great time to head out.  

**Quick checklist before you go:**

| Factor | What to look for | Why it matters |
|--------|------------------|----------------|
| **Temperature** | Aim for 50‑70 °F (10‑21 °C) for most people. | Extreme heat or cold can make a walk uncomfortable or unsafe. |
| **Humidity** | Moderate humidity (40‑60 %) feels best. | High humidity can make it feel hotter and more tiring. |
| **Wind** | Light breezes (under 10 mph) are fine; strong gusts can be uncomfortable. | Strong wind can make it feel colder and affect balance. |
| **Air quality** | Check the AQI; values under 50 are “good.” | Poor air quality can irritate lungs, especially if you have asthma. |
| **Sun exposure** | If it’s bright, consider sunscreen, a hat, or sunglasses. | Prote